# ETL Exploration
Use this notebook to prototype and test your ETL pipeline before moving code to `etl/etl.py`.

In [2]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()
SIMFIN_API_KEY = os.getenv('SIMFIN_API_KEY')
print('API key loaded:', bool(SIMFIN_API_KEY))

API key loaded: True


## 1. Fetch raw data
TODO: call PySimFin or simfin library here.

In [3]:
# TODO
df = pd.read_csv('../data/raw/us-shareprices-daily.csv', sep=';')

In [ ]:
#Choosing our five companies
AMZN = df[df['Ticker'] == 'AMZN']
MSFT = df[df['Ticker'] == 'MSFT']
PLTR = df[df['Ticker'] == 'PLTR']
GOOG = df[df['Ticker'] == 'GOOG']
AAPL = df[df['Ticker'] == 'AAPL']

## 2. Clean & explore
Check for nulls, data types, date ranges.

In [9]:
df.shape

(6201544, 11)

In [12]:
df.dtypes

Ticker                 object
SimFinId                int64
Date                   object
Open                  float64
High                  float64
Low                   float64
Close                 float64
Adj. Close            float64
Volume                  int64
Dividend              float64
Shares Outstanding    float64
dtype: object

In [13]:
df.isnull().sum()

Ticker                    667
SimFinId                    0
Date                        0
Open                        0
High                        0
Low                         0
Close                       0
Adj. Close                  0
Volume                      0
Dividend              6165309
Shares Outstanding     525171
dtype: int64

In [16]:
#Check date range
print(f'Min date: {df.Date.min()}, Max date: {df.Date.max()}')

Min date: 2020-04-06, Max date: 2025-03-08


In [21]:
companies_df = pd.read_csv('../data/raw/us-companies.csv', sep=';')
companies_df.head()

,Ticker,SimFinId,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,Market,CIK,Main Currency
0,NaN,18692750,NaN,NaN,NaN,NaN,NaN,NaN,us,1997711.0,USD
1,NaN,18847915,NaN,NaN,NaN,NaN,NaN,NaN,us,1769731.0,USD
2,NaN,18538670,NaN,NaN,NaN,NaN,NaN,NaN,us,1734107.0,USD
3,NaN,18657366,NaN,NaN,NaN,NaN,NaN,NaN,us,1899830.0,USD
4,NaN,18667300,NaN,NaN,NaN,NaN,NaN,NaN,us,1178819.0,USD


In [24]:
companies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6526 entries, 0 to 6525
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Ticker                         6489 non-null   object 
 1   SimFinId                       6526 non-null   int64  
 2   Company Name                   6492 non-null   object 
 3   IndustryId                     6224 non-null   float64
 4   ISIN                           5344 non-null   object 
 5   End of financial year (month)  6493 non-null   float64
 6   Number Employees               5701 non-null   float64
 7   Business Summary               6232 non-null   object 
 8   Market                         6526 non-null   object 
 9   CIK                            6514 non-null   float64
 10  Main Currency                  6526 non-null   object 
dtypes: float64(4), int64(1), object(6)
memory usage: 561.0+ KB


## Building Data Cleansing Functions

**Why?** We want to avoid redundancy and being more time-efficient

In [ ]:
TICKERS = ['AAPL', 'PLTR', 'GOOG', 'AMZN', 'MSFT']

def clean(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['Dividend'] = df['Dividend'].fillna(0)
    df = df.drop(columns=['SimFinId'])
    return df

## 3. Feature engineering
Create the derived columns the model will use.

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['Return'] = df['Close'].pct_change()
    df['MA5'] = df['Close'].rolling(5).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['Volume_Change'] = df['Volume'].pct_change()
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    df = df.dropna().reset_index(drop=True)
    return df

processed = {}
for ticker in TICKERS:
    company_df = df[df['Ticker'] == ticker]
    cleaned = clean(company_df)
    featured = engineer_features(cleaned)
    featured.to_csv(f'../data/processed/{ticker}.csv', index=False)
    processed[ticker] = featured
    print(f'{ticker}: {len(featured)} rows saved')